# KrishiDisha teacher distillation on a Kaggle CPU kernel

Runs the teacher pool (Groq, Gemini, OpenRouter, optional Ollama cloud) for the trajectory and Hindi-rewrite stages, resuming from the private cache dataset and re-uploading the cache after every pass, for up to `MAX_HOURS`. Secrets needed (Add-ons > Secrets): `GEMINI_API_KEY`, `GROQ_API_KEY`, `OPENROUTER_API_KEY`, `KAGGLE_USERNAME`, `KAGGLE_KEY` (to version the cache dataset), optional `OLLAMA_API_KEY`.


In [ ]:
# Source comes from the private Kaggle dataset (git archive of the lab repo), not from GitHub.
# Kaggle extracts uploaded zips, so the tree is at .../krishidisha-lab-src/src/krishidisha; fall back to src.zip.
import glob, os, shutil, zipfile
shutil.rmtree("/tmp/krishidisha", ignore_errors=True)
_dirs = glob.glob("/kaggle/input/**/krishidisha-lab-src/src/krishidisha", recursive=True)
if _dirs:
    shutil.copytree(_dirs[0], "/tmp/krishidisha")
    _ver = os.path.join(os.path.dirname(os.path.dirname(_dirs[0])), "VERSION.txt")
else:
    _zip = glob.glob("/kaggle/input/**/krishidisha-lab-src/**/src.zip", recursive=True)[0]
    zipfile.ZipFile(_zip).extractall("/tmp")
    _ver = os.path.join(os.path.dirname(_zip), "VERSION.txt")
print("source version:", open(_ver).read().strip() if os.path.exists(_ver) else "unknown", "->", os.listdir("/tmp/krishidisha")[:8])

MAX_HOURS = 11.0
PROVIDERS = "groq,gemini,openrouter"      # + ",ollama" when OLLAMA_API_KEY is set (Ollama cloud through ollama.com)
import os, sys, subprocess, shutil, glob, time, json
W = "/kaggle/working"
os.chdir("/tmp/krishidisha")
from kaggle_secrets import UserSecretsClient
_s = UserSecretsClient()
for k in ["GEMINI_API_KEY", "GROQ_API_KEY", "OPENROUTER_API_KEY", "KAGGLE_USERNAME", "KAGGLE_KEY", "OLLAMA_API_KEY"]:
    try:
        os.environ[k] = _s.get_secret(k)
    except Exception:
        print("secret missing:", k)
if os.environ.get("OLLAMA_API_KEY"):
    os.environ["TEACHER_OLLAMA_BASE_URL"] = "https://ollama.com/v1"
    PROVIDERS += ",ollama"
os.environ["TRANSLATION_BACKEND"] = "none"
os.environ["KB_EMBEDDING_MODEL"] = ""
print("providers:", PROVIDERS)


In [ ]:
CMD = "pip install -q flask flask-sqlalchemy python-dotenv openai anthropic datasketch reportlab 2>&1 | tail -1"
!{CMD}


In [ ]:
# seed data + cache from the cache dataset (Kaggle extracted the zip)
src_root = [p for p in glob.glob("/kaggle/input/**/krishidisha-distill-cache", recursive=True)][0]
os.makedirs("data/llm/.cache", exist_ok=True)
for p in glob.glob(src_root + "/**/*.jsonl", recursive=True):
    dst = "data/llm/.cache/" + os.path.basename(p) if "/cache/" in p.replace(chr(92), "/") else "data/llm/" + os.path.basename(p)
    shutil.copy2(p, dst)
print({os.path.basename(p): sum(1 for _ in open(p, encoding="utf-8")) for p in glob.glob("data/llm/.cache/*.jsonl")})
print("questions:", sum(1 for _ in open("data/llm/questions.jsonl", encoding="utf-8")))


In [ ]:
def push_cache(note):
    """Version the cache dataset from inside the kernel (needs KAGGLE_USERNAME/KAGGLE_KEY secrets)."""
    if not os.environ.get("KAGGLE_KEY"):
        print("no KAGGLE_KEY secret: cache stays in /kaggle/working only"); return
    tmp = "/tmp/cache_upload"; shutil.rmtree(tmp, ignore_errors=True); os.makedirs(tmp + "/cache")
    for p in glob.glob("data/llm/.cache/*.jsonl"):
        shutil.copy2(p, tmp + "/cache/")
    for n in ["questions.jsonl", "train.jsonl", "eval.jsonl", "kisanvaani.jsonl", "kb_pairs.jsonl"]:
        if os.path.exists("data/llm/" + n):
            shutil.copy2("data/llm/" + n, tmp + "/")
    json.dump({"title": "KrishiDisha distillation cache", "id": "sspanwar/krishidisha-distill-cache",
               "licenses": [{"name": "other"}]}, open(tmp + "/dataset-metadata.json", "w"))
    r = subprocess.run(["kaggle", "datasets", "version", "-p", tmp, "-m", note, "--dir-mode", "zip"], capture_output=True, text=True)
    print("cache push:", (r.stdout + r.stderr).strip()[-200:])

def counts():
    return {os.path.basename(p): sum(1 for _ in open(p, encoding="utf-8")) for p in sorted(glob.glob("data/llm/.cache/*.jsonl"))}

t0 = time.time(); passes = 0
while time.time() - t0 < MAX_HOURS * 3600:
    passes += 1
    before = counts()
    for stage in [f"python -u -m ml.llm.distill trajectories --provider {PROVIDERS} --questions data/llm/questions.jsonl --out {W}/distill.jsonl",
                  f"python -u -m ml.llm.distill rewrite --provider {PROVIDERS} --inp data/llm/train.jsonl --filter-language hi --target-language hi --out {W}/kb_hi_rewritten.jsonl"]:
        r = subprocess.run(stage + f" 2>&1 | grep -vE 'INFO|Warning' | tee -a {W}/distill.log | tail -8", shell=True)
    after = counts()
    print(f"pass {passes} at {(time.time()-t0)/3600:.1f}h: {after}", flush=True)
    for p in glob.glob(f"{W}/*.jsonl"):
        pass
    shutil.copytree("data/llm/.cache", f"{W}/cache", dirs_exist_ok=True)
    push_cache(f"pass {passes}: {after}")
    if after == before:
        # nothing new: every provider is out of daily quota; wait for the next reset (Google/Groq reset at 00:00 PT)
        wait = 1800
        if time.time() - t0 + wait > MAX_HOURS * 3600:
            break
        print(f"no progress; sleeping {wait//60} min", flush=True); time.sleep(wait)
print("done:", counts())
